<img src='https://gitlab.eumetsat.int/eumetlab/oceans/ocean-training/tools/frameworks/-/raw/main/img/MSOC_banner.png' align='right' width='100%'/>

<a href="../Index.ipynb">Index</a>

**Copyright:** 2025 MSOC training team <br>
**License:** MIT <br>
**Authors:**  Tom Jackson (EUMETSAT), Ben Loveday (EUMETSAT/Innoflair UG), Hayley Evers-King (EUMETSAT), Vittorio Brando (CNR)

<div class="alert alert-block alert-success">
<h3>Multi-sensor Ocean Colour Course</h3></div>

<div class="alert alert-block alert-warning">
    
<b>PREREQUISITES </b>

This notebook has the following prerequisites:
- **<a href="https://data.marine.copernicus.eu/register" target="_blank">A Copernicus Marine Service account</a>** to download data form the Copernicus Marine Service Data Store
- You should have built and activated the appropriate `msoc` Python environment in either your command line, or in the Anaconda navigator prior to launching this notebook.

There are no prerequisite notebooks for this module.
</div>
<hr>

# Comparing single-sensor and multi-sensor level-3 ocean colour products

### Data used

| Dataset | Copernicus Marine<br>Data Store product ID | Copernicus Marine<br>product description | WEkEO dataset ID | WEkEO description |
|:--------------------:|:-----------------------:|:-------------:|:-----------------:|:-----------------:|
| Global Ocean Colour (Copernicus-GlobColour), Bio-Geo-Chemical, L3 (daily) from Satellite Observations (Near Real Time) | OCEANCOLOUR_GLO_BGC_L3_NRT_009_101 | <a href="https://data.marine.copernicus.eu/product/OCEANCOLOUR_GLO_BGC_L3_NRT_009_101/description" target="_blank">Description</a> | OCEANCOLOUR_GLO_BGC_L3_NRT_009_101 | <a href="https://www.wekeo.eu/data?view=dataset&dataset=EO%3AMO%3ADAT%3AOCEANCOLOUR_GLO_BGC_L3_NRT_009_101" target="_blank">Description</a> |
| North Atlantic Ocean Colour Plankton, Reflectance, Transparency and Optics, L3 (daily) from Satellite Observations (Near Real Time) | OCEANCOLOUR_ATL_BGC_L3_NRT_009_111 | <a href="https://data.marine.copernicus.eu/product/OCEANCOLOUR_ATL_BGC_L3_NRT_009_111/description" target="_blank">Description</a> | OCEANCOLOUR_ATL_BGC_L3_NRT_009_111 | <a href="https://www.wekeo.eu/data?view=dataset&dataset=EO%3AMO%3ADAT%3AOCEANCOLOUR_ATL_BGC_L3_NRT_009_111" target="_blank">Description</a> |

### Learning outcomes

At the end of this notebook you will know;
* How to compare multi-sensor records to determine their characteristics and stability

### Outline

When working with multi-sensor records it is important to understand the characteristics of your data, especially if the sensors that are used to create the record have changed over time. In this notebook, we will take a look at some of these records to see how their resolutions and length influence their ability to render spatial and temporal variability. We will also look at some of these records and assess their stability to see if we can isolate any changes that may be associated with shifts in sensors, changes in coverage.

<hr>

<div class="alert alert-info" role="alert">

## Importing dependencies

</div>

We begin by importing all of the libraries that we need to run this notebook. If you have built your python using the environment file provided in this repository, then you should have everything you need. For more information on building environment, please see the repository **<a href="../README.md" target="_blank">README</a>**.

In [ ]:
import copernicusmarine             # a library to help us access CMEMS data
import eumartools                   # a EUMETSAT library that support working with Copernicus Sentinel products
import os                           # a library that allows us access to basic operating system commands like making directories
from pathlib import Path            # a library that helps construct system path objects
import datetime                     # a library that allows us to work with dates and times
import xarray as xr                 # a library that supports the use of multi-dimensional arrays in Python
import numpy as np                  # a library that provides support for array-based mathematics
import matplotlib                   # a library that support plotting
import matplotlib.pyplot as plt     # a library that support plotting
import matplotlib.ticker as mticker # a library that extends plotting support
import cartopy                      # a library that supports mapping and projection
import cartopy.crs as ccrs          # a library that support mapping
import stats_tests                  # a bespoke library for trend analysis
import logging                      # a library that provides log level management
import warnings                     # a library that helps us manage warnings
import dask                         # a library for parallel computing on large datasets efficiently
from scipy.stats import shapiro, normaltest

logging.getLogger("copernicusmarine").setLevel("ERROR")

We will also set a few parameters to making running this notebook more convenient.

In [ ]:
# suppress warnings
warnings.filterwarnings('ignore')

# memory management
dask.config.set({"array.slicing.split_large_chunks": True})
chunks={"rows": 512, "columns": 512}

# plotting control
matplotlib.rcParams.update({'font.size': 14})
yticks = [-2, -1, 0, 1, 2]

<div class="alert alert-warning" role="alert">

## Defining functions

</div>

Next we will define a some quick functions for re-use below. These will be used to embellish any output plots. These cells are hidden by default but you can click on the arrow next to this cell to expand them.

In [ ]:
def embellish_plot(m):
    """
    Add land features and labeled gridlines to a cartopy axis.

    Parameters
    ----------
    m : cartopy.mpl.geoaxes.GeoAxes
        Axis to embellish.

    Returns
    -------
    None
    """
    # Embellish with gridlines
    m.add_feature(cartopy.feature.NaturalEarthFeature('physical', 'land', '10m', edgecolor='k', facecolor='#7E9CA0', linewidth=0.5), zorder=500)
    g1 = m.gridlines(draw_labels = True, linestyle='--', linewidth=0.5, zorder=1000)
    g1.top_labels = g1.right_labels = False
    g1.xlabel_style = g1.ylabel_style = {'color': '0.5'}

In [ ]:
def annotate_plot(ax, mean, std, log=True):
    ax.axvline(mean,  ls="--", c='black')
    ax.set_title(f'{d_test.title} stats 1', fontsize=10)
    ax.set_ylim([0, 80])
    ax.set_xlabel("CHL [mg.$m^{-3}$]")
    ax.set_ylabel("Counts")
    if (log==True):
        xticks = [-2, -1, 0, 1, 2]
        ax.set_xlim([-2, 2])
        ax.set_xticks(xticks, [10**i for i in xticks])
        ax.annotate(f'Mean: {10**mean:.2f}\nSTD(logchl): {std:.2f}', (-1.9, 10), c="r")
    else:
        ax.annotate(f'Mean: {mean:.2f}\nSTD: {std:.2f}', (-1.0, 10), c="r")

<div class="alert alert-info" role="alert">

## Setting up the experiment

</div>

Next we will create a download directory to store the products we will download in this notebook.

In [ ]:
download_dir = os.path.join(os.path.dirname(os.getcwd()), "Data", "Preprepared", "Day4", "Multi_sensor_comparison")
os.makedirs(download_dir, exist_ok=True)

Lets now, specify the Copernicus Marine Service products that we want to work with. In the example case, we will choose four products;
* the Global 4 km multi-sensor multi-year product
* the Atlantic 1 km multi-sensor multi-year product
* the Global 300 m OLCI multi-year product
* the Atlantic 300 m OLCI near real-time product

In [ ]:
products = []
products.append({"region" : "glo", "product" : "plankton", "rec": "my", "sensors" : "multi", "resolution" : "4km", "timing" : "1D"})
products.append({"region" : "atl", "product" : "plankton", "rec": "my", "sensors" : "multi", "resolution" : "1km", "timing" : "1D"})
products.append({"region" : "glo", "product" : "plankton", "rec": "my", "sensors" : "olci", "resolution" : "300m", "timing" : "1D"})
products.append({"region" : "atl", "product" : "plankton", "rec": "nrt", "sensors" : "olci", "resolution" : "300m", "timing" : "1D"})

We will select the chlorophyll product from each.

In [ ]:
CMEMS_variables = ['CHL']

Lets select our spatial and temporal region of interest. Note that we are selecting an "NRT" product, so we need to be sure that we only look at data available in the NRT "era"...

*Note: be careful when you adapt this, your NRT variability may change!*

In [ ]:
# space filter for matching products
west = -7
south = 47.5
east = -3.5
north = 51

# time filter for matching products
dtstart = datetime.datetime(2024, 1, 1, 0, 0)
dtend = datetime.datetime(2025, 11, 13, 0, 0)

We are also going to look at a much longer time series for the multi-year (my) products. TO make sure we don't violate the data limits of the `copernicusmarine` toolkit, we will choose a second, smaller, area to look at, in this case.

In [ ]:
# space filter for matching products
west_long = 3.
south_long = 51.
east_long = 4.
north_long = 52.

# time filter for matching products
dtstart_long = datetime.datetime(1998, 1, 1, 0, 0)
dtend_long = datetime.datetime(2025, 11, 13, 0, 0)

<div class="alert alert-info" role="alert">

## Downloading level-3 ocean colour products from the Copernicus Marine Service

</div>

Lets now get our data, starting by creating a credentials file to support authorisation of the Copernicus Marine Service. Please see the notebooks from Day 1 of the course if you need help with this step.

In [ ]:
# Default location expected by the copernicusmarine package
copernicus_marine_credentials_file = Path(Path.home() / '.copernicusmarine' / '.copernicusmarine-credentials')

# Create it only if it does not already exists
if not copernicus_marine_credentials_file.is_file():
    copernicusmarine.login()

Next we will cycle through our products of interest, downloading all the data to our specified directory.

In [ ]:
output_filenames = []
for product in products:
    CMEMS_product = f"cmems_obs-oc_{product['region']}_bgc-{product['product']}_{product['rec']}_l3-{product['sensors']}-{product['resolution']}_P{product['timing']}"
    output_filename=os.path.join(os.getcwd(), download_dir,
        f"{CMEMS_product}_{dtstart.strftime('%Y%m%d')}_{dtend.strftime('%Y%m%d')}_{str(west)}_{str(south)}_{str(east)}_{str(north)}.nc")
    output_filenames.append(output_filename)
    
    if os.path.exists(output_filename):
        print(f"Skipping: {CMEMS_product} as it exists")
        continue
    else:
        print(f"Fetching: {CMEMS_product}")
        copernicusmarine.subset(
            dataset_id=CMEMS_product,
            variables=CMEMS_variables,
            minimum_longitude=west,
            maximum_longitude=east,
            minimum_latitude=south,
            maximum_latitude=north,
            start_datetime=dtstart.strftime("%Y-%m-%dT00:00:00.000Z"),
            end_datetime=dtend.strftime("%Y-%m-%dT00:00:00.000Z"),
            output_filename=output_filename);

<div class="alert alert-info" role="alert">

## Comparing level-3 ocean colour products from the Copernicus Marine Service

</div>

Assuming the above cells all worked well, now we can open our datasets and start to compare them...

In [ ]:
datasets = []
for output_filename in output_filenames:
    datasets.append(xr.open_dataset(output_filename, chunks=chunks))

First, lets plot the standard deviation of each product and see what differences we can see.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(18, 15), dpi=300, subplot_kw={"projection": ccrs.PlateCarree()})
for ds, ax in zip(datasets, axs.flatten()):
    p1 = ds["CHL"].std(dim=["time"]).plot(ax=ax, vmin=0, vmax=5, cbar_kwargs={"label": "std.dev CHL [mg.m$^{-3}$]"})
    ax.set_title(ds.title, fontsize=10)
    embellish_plot(ax)

<div class="alert alert-danger" role="alert">

**WAIT! What do you notice about the above?**

* How do the products compare?
* What information do you gain from increasing spatial resolution?
* Do you notice any differences between the multi-year and nrt product (bottom panels)?
  
</div>

Lets look at the data in a different way, comparing the scene averaged value over time...

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 10), dpi=150)

labels = []
traces = []
for ds, itercol in zip(datasets, np.linspace(0, 1, len(datasets)+1)):
    cvalue = plt.cm.get_cmap("viridis")(itercol)
    p1, = np.log10(ds["CHL"]).mean(dim=["latitude", "longitude"]).plot(c=cvalue , linewidth=0.5, linestyle="--", marker="o", markersize=6, alpha=0.25)
    labels.append(ds.title)
    traces.append(p1)

ax.set_yticks(yticks, [10**i for i in yticks])
ax.set_ylim([-1, 1.5])
ax.set_xlabel("Date")
ax.set_ylabel("CHL [mg.$m^{-3}$]")
leg = ax.legend(traces, labels)
leg.set_frame_on(False)

Where do you notice clear differences between the products? Can you plot these time slices and compare?

Lets look at the distibution of values and some summary statistics for the area of interest for one of the 4 datasets.

In [ ]:
d_test=datasets[0]
fig, axs = plt.subplots(1, 3, figsize=(20, 8), dpi=150)
xticks = [-2, -1, 0, 1, 2]
ax = axs.flatten()[0] 
p1 = ax.hist(d_test["CHL"].mean(dim=["latitude", "longitude"]), bins=np.arange(-2, 2, 0.05))
mean_1=np.nanmean(d_test["CHL"].mean(dim=["latitude", "longitude"]))
std_1=np.nanstd(d_test["CHL"].mean(dim=["latitude", "longitude"]))
annotate_plot(ax, mean_1, std_1, log=False)

ax = axs.flatten()[1] 
p2 = ax.hist(np.log10(d_test["CHL"].mean(dim=["latitude", "longitude"])), bins=np.arange(-2, 2, 0.05))
mean_2=np.nanmean(np.log10(d_test["CHL"].mean(dim=["latitude", "longitude"])))
std_2=np.nanstd(np.log10(d_test["CHL"].mean(dim=["latitude", "longitude"])))
annotate_plot(ax, mean_2, std_2, log=True)

ax = axs.flatten()[2] 
p3 = ax.hist(np.log10(d_test["CHL"]).mean(dim=["latitude", "longitude"]), bins=np.arange(-2, 2, 0.05))
mean_3=np.nanmean(np.log10(d_test["CHL"]).mean(dim=["latitude", "longitude"]))
std_3=np.nanstd(np.log10(d_test["CHL"]).mean(dim=["latitude", "longitude"]))
annotate_plot(ax, mean_3, std_3, log=True)

<div class="alert alert-danger" role="alert">

**WAIT! What do you notice about the above?**

* So which panel has the most 'correct' usage of statistics?  Left, Middle or right?
  
</div>

You might have heard of some tests you can do for 'normality' in a dataset (meaning parametric stats like mean are valid and can be interpreted sensibly). Tests include looking at histograms of data, Q-Q plots and statistical tests like shapiro-wilks or D’Agostino–Pearson K². Lets try a little test (Shapiro-wilks) to see if the data distribution is normal or not...

In [ ]:
chl_flat=np.ravel(d_test["CHL"])
chl_flat_valid=chl_flat[~np.isnan(chl_flat)]
fig, axs = plt.subplots(2, 1, figsize=(20, 8), dpi=150)
ax=axs.flatten()[0]
ax.hist(chl_flat_valid, bins=np.arange(0, 30, 0.1))
shap=shapiro(chl_flat_valid)
n_test=normaltest(chl_flat_valid)
ax.annotate(f'Shapiro-Wilks test value: {shap.statistic:.2f}', xy=(0.6, 0.4), xycoords='axes fraction')
ax.annotate(f'Shapiro-Wilks test p-val: {shap.pvalue:.6f}', xy=(0.6, 0.3), xycoords='axes fraction')
if shap.pvalue < 0.05:
    ax.annotate('Shapiro-Wilks test: Not Normal', xy=(0.6, 0.2), xycoords='axes fraction')
else :
    ax.annotate('Shapiro-Wilks test: Normal', xy=(0.6, 0.2), xycoords='axes fraction')

ax=axs.flatten()[1]
ax.hist(np.log10(chl_flat_valid), bins=np.arange(-2,2, 0.05))
shap=shapiro(np.log10(chl_flat_valid))
ax.annotate(f'Shapiro-Wilks test value: {shap.statistic:.2f}', xy=(0.6, 0.4), xycoords='axes fraction')
ax.annotate(f'Shapiro-Wilks test p-val: {shap.pvalue:.6f}', xy=(0.6, 0.3), xycoords='axes fraction')
if shap.pvalue < 0.05:
    ax.annotate('Shapiro-Wilks test: Not Normal', xy=(0.6, 0.2), xycoords='axes fraction')
else :
    ax.annotate('Shapiro-Wilks test: Normal', xy=(0.6, 0.2), xycoords='axes fraction')

Despite a value very close to 1 the second distribution is rejected as normal due to the tiny p-value.  This is due to a very high sensitivity of this test at large n (the test is not really suitable for n>5000). In this case the shapiro test is not the best tool, but we can clearly see that the second distribution is closer to normal than the first (and this is reflected in the test value). Now we have established that the chlorophyll data in this area are roughly log normal, let us proceed with our investigations on the charactistics of the chlorophyll-a data for this region for each of the 4 datasets.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 8), dpi=150)
xticks = [-2, -1, 0, 1, 2]
for ds, ax, itercol in zip(datasets, axs.flatten(), np.linspace(0, 1, len(datasets)+1)):
    cvalue = plt.cm.get_cmap("viridis")(itercol)    
    p1 = ax.hist(np.log10(ds["CHL"]).mean(dim=["latitude", "longitude"]), bins=np.arange(-2, 2, 0.05), color=cvalue)
    mean=10**np.nanmean(np.log10(ds["CHL"]).mean(dim=["latitude", "longitude"]))
    ax.axvline(np.log10(mean),  ls="--", c='black')
    ax.set_title(ds.title, fontsize=10)
    ax.set_xlim([-2, 2])
    ax.set_xticks(xticks, [10**i for i in xticks])
    ax.set_ylim([0, 80])
    ax.set_xlabel("CHL [mg.$m^{-3}$]")
    ax.set_ylabel("Counts")
    ax.annotate(f'Mean: {mean:.2f}\nSTD (logchl): {np.nanstd(np.log10(ds["CHL"]).mean(dim=["latitude", "longitude"])):.2f}', (-1.9, 10), c="r")

Here we can see subtle differences between the statistics of the 4 datasets.  One would assume the regional product is likely the most suitable for any given area but what happens if your area of interest straddles the boudary of two regional datasets?

Please feel free to spend a little longer performing your own comparative anaylses of the datasets for this area.  How does the coverage vary between the datasets?  

Also this brief investigation has been over a relatively small time series, lets take a look at the differences between potential datasets over a longer time-series.

<div class="alert alert-info" role="alert">

## Investigating trends in level-3 ocean colour products

</div>

First we need to retrieve some long records, we will choose a smaller geographic region to keep total volumes small.

In [ ]:
output_filenames_long = []
for product in products:
    if "nrt" in product['rec']:
        print("Skipping nrt record")
        continue

    CMEMS_product = f"cmems_obs-oc_{product['region']}_bgc-{product['product']}_{product['rec']}_l3-{product['sensors']}-{product['resolution']}_P{product['timing']}"
    output_filename_long = os.path.join(os.getcwd(), download_dir,
        f"{CMEMS_product}_{dtstart_long.strftime('%Y%m%d')}_{dtend_long.strftime('%Y%m%d')}_{str(west_long)}_{str(south_long)}_{str(east_long)}_{str(north_long)}_long.nc")

    output_filenames_long.append(output_filename_long)

    if os.path.exists(output_filename_long):
        print(f"Skipping: {CMEMS_product} as it exists")
        continue
    else:
        print(f"Fetching: {CMEMS_product}")
        copernicusmarine.subset(
            dataset_id=CMEMS_product,
            variables=CMEMS_variables,
            minimum_longitude=west_long,
            maximum_longitude=east_long,
            minimum_latitude=south_long,
            maximum_latitude=north_long,
            start_datetime=dtstart_long.strftime("%Y-%m-%dT00:00:00.000Z"),
            end_datetime=dtend_long.strftime("%Y-%m-%dT00:00:00.000Z"),
            output_filename=output_filename_long);

Load data

In [ ]:
datasets_long = []
for output_filename_long in output_filenames_long:
    datasets_long.append(xr.open_dataset(output_filename_long, chunks=chunks))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 10), dpi=300)

labels = []
traces = []
for ds, itercol in zip(datasets_long, np.linspace(0, 1, len(datasets)+1)):
    cvalue = plt.cm.get_cmap("turbo")(itercol*2)
    p1, = np.log10(ds["CHL"]).mean(dim=["latitude", "longitude"]).plot(c=cvalue, linewidth=0.5, linestyle="-", marker="+", markersize=6, alpha=0.5)
    labels.append(ds.title)
    traces.append(p1)

ax.set_yticks(yticks, [10**i for i in yticks])
ax.set_ylim([-1, 2])
ax.set_xlabel("Date")
ax.set_ylabel("CHL [mg.$m^{-3}$]")
leg = ax.legend(traces, labels)
leg.set_frame_on(False)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20, 8), dpi=150)
xticks = [-2, -1, 0, 1, 2]
for ds, ax, itercol in zip(datasets_long, axs.flatten(), np.linspace(0, 1, len(datasets)+1)):
    cvalue = plt.cm.get_cmap("plasma")(itercol)    
    p1 = ax.hist(np.log10(ds["CHL"]).mean(dim=["latitude", "longitude"]), bins=np.arange(-2, 2, 0.05), color=cvalue)
    ax.axvline(np.mean(np.log10(ds["CHL"]).mean(dim=["latitude", "longitude"])),  ls="--", c='black')
    ax.set_title(ds.title, fontsize=10)
    ax.set_xlim([-2, 2])
    ax.set_xticks(xticks, [10**i for i in xticks])
    ax.set_xlabel("CHL [mg.$m^{-3}$]")
    ax.set_ylabel("Counts")
    ax.annotate(f'Mean: {10**np.nanmean(np.log10(ds["CHL"])):.2f}\nSTD(logchl): {np.nanstd(np.log10(ds["CHL"])):.2f}', (-1.9, 10), c="r")

Are the datasets consistent over this longer time series?

It looks like the 300m dataset has a significant offset (bias) relative to the the two 'multi' datasets.  This doesnt mean that the 300m dataset is wrong (the other datasets could have had olci corrected back to another reference for example).  Validation of the two datasets in the latter part of the time series could elucidate which could be considered a better representation of the in-water values.

But what about looking for trends in the datasets (irrespective of the correctness of the absolute values)?

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 10), dpi=150)

labels = []
traces = []
for ds, itercol in zip(datasets_long, np.linspace(0, 1, len(datasets)+1)):

    trend=stats_tests.detect_trend(np.log10(ds["CHL"]).mean(dim=["latitude", "longitude"]))
    t = ds.time.values
    t_years = (t - t[0]) / np.timedelta64(365, 'D')
    trend_line = trend[0][0] + trend[0][1] * t_years
    
    cvalue = plt.cm.get_cmap("plasma")(itercol)
    p1, = np.log10(ds["CHL"]).mean(dim=["latitude", "longitude"]).plot(c=cvalue, linewidth=0.5, linestyle="--", marker="o", markersize=6, alpha=0.25)
    traces.append(p1)
    labels.append(ds.title)

    p2, = plt.plot(t, trend_line, c="w", linewidth=4, zorder=3)
    p2, = plt.plot(t, trend_line, c=cvalue, linestyle = "--", zorder=4)
    labels.append(ds.title+'_trend')
    traces.append(p2)

ax.set_yticks(yticks, [10**i for i in yticks])
ax.set_ylim([-1, 2])
ax.set_xlabel("Date")
ax.set_ylabel("CHL [mg.$m^{-3}$]")
leg = ax.legend(traces, labels)
leg.set_frame_on(False)

Note the difference in the trends between the single sensor and multi sensor records.

Let take a look at the longer 4km global time series and see if it is a single trend or the time series has a break point in it.

In [ ]:
ds = datasets_long[0]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 10), dpi=150)

ts = np.log10(ds["CHL"]).mean(dim=["latitude", "longitude"])
seasonal_weekly = ts.groupby("time.week").mean("time")
anomalies_weekly = ts - seasonal_weekly.sel(week=ts.time.dt.week)
anomalies_weekly.plot(c="#1C69A8")
ax.set_title('Anomalies relative to weekly climatology')
ax.set_xlabel("Date")
ax.set_ylabel("log10(CHL) anomaly [mg.$m^{-3}$]")

In [ ]:
break_idx, segment_trends = stats_tests.detect_trend_segments(anomalies_weekly, max_segments=2)

In [ ]:
trend = stats_tests.detect_trend(anomalies_weekly)
trend2 = stats_tests.detect_trend(anomalies_weekly.isel(time=slice(0,int(break_idx[0]))))
trend3 = stats_tests.detect_trend(anomalies_weekly.isel(time=slice(int(break_idx[0]),len(anomalies_weekly))))
# create axes first

# compute single trend line
t = anomalies_weekly.time.values
t_years = (t - t[0]) / np.timedelta64(365, 'D')
trend_line = trend[0][0] + trend[0][1] * t_years
trend_line2 = trend2[0][0] + trend2[0][1] * t_years[0:int(break_idx[0])]
trend_line3 = trend3[0][0] + trend3[0][1] * t_years[0:len(anomalies_weekly)-int(break_idx[0])]

In [ ]:
# overlay on the same axes
fig, ax = plt.subplots(1, 1, figsize=(20, 10), dpi=150)

ax.plot(t, anomalies_weekly, c="#1C69A8")
ax.plot(t, trend_line, linewidth=2, label='Single Trend', color='red')
ax.plot(t[:int(break_idx[0])], trend_line2, linewidth=2, label='Double Trend 1', color='yellow')
ax.plot(t[int(break_idx[0]):], trend_line3, linewidth=2, label='Double Trend 2', color='orange')

ax.set_ylim([-1, 1])
ax.set_xlabel("Date")
ax.set_ylabel("CHL anomaly [mg.$m^{-3}$]")
leg = ax.legend()
leg.set_frame_on(False)

<div class="alert alert-info" role="alert">

## Is the breakpoint real and/or significant?

</div>

The Akaike information criterion (AIC) deals with the trade-off between the goodness of fit of the model and the simplicity of the model. In other words, AIC deals with both the risk of overfitting and the risk of underfitting.

We can also consider the Bayesian information criterion (BIC) which is similar to the AIC but generally penalizes free parameters more strongly, though it depends on the size of n and relative magnitude of n (points) and k (parameters).

In [ ]:
results = stats_tests.test_breakpoint_significance(anomalies_weekly, break_idx[0])

for k, v in results.items():
    print(f"{k}: {v}")

Though the F-test gives p ≈ 0.084 the AIC is almost identical for the 2-segment model. So the p-value is not < 0.05 (a common threshold for hypothesis testing).  Even if the p-value was slightly lower, as with the earlier test this can be deceptive in long time series as even a very small reduction in RSS can be statistically “significant” because the test accounts for sample size. With thousands of points, the F-test is extremely sensitive.

Given the extremely minor relative change in the RSS and AIC between the two trend models it looks like there is no grounds to say there is a significant break in this time series.

<div class="alert alert-info" role="alert">

## Interpretation

</div>

Statistical significance vs practical significance

p < 0.05 → statistically, the 2-segment model fits better

But Residual sum of squares (RSS) improvement is very small → in practical terms, the second trend is almost the same as the first

AIC/BIC differences are minor → the extra complexity may not be meaningful

Rule of thumb

If RSS reduction is <1% of total variance, even a significant p-value may indicate overfitting small noise, not a real break

Look at the trend lines visually — if they’re nearly parallel, the break is probably artificial

Conclusion in your case

Statistically significant (F-test), but the effect size is tiny

Likely the “break” is being introduced by the test picking up very small deviations, not a true change in trend

<div class="alert alert-success" role="alert">

## What next?

Now go do some similar investigations in your area.  If you finish a provisional trend analysis why not try to characterise your regions phenology and look for changes in phenological indicators?  Note, this is not a simple task, if someone in your group does not have expertise with phenology then feel free to ask for info from the instructors.

</div>

<hr>
<a href="../Index.ipynb">Index</a>
<hr>
<a href="https://gitlab.com/eo_training/msoc" target="_blank">View on GitLab</a> | <a href="https://training.eumetsat.int/" target="_blank">EUMETSAT Training</a> | <a href=mailto:ops@eumetsat.int target="_blank">Contact helpdesk for support </a> | <a href=mailto:training@eumetsat.int target="_blank">Contact our training team to collaborate on and reuse this material</a></span></p>